# Gradients from scratch

A derivative can always be measured without calculus: change the input by a
small step $h$, evaluate the function again, and divide the change in output
by $h$. This notebook derives gradients with the rules from the chapter and
compares each one against that measurement. An `assert` stops the notebook if
any comparison fails, so a clean run means every formula below is correct.

Matching sections of [03_calculus.md](03_calculus.md):

- §2.1 The derivative is local sensitivity: forward vs centered difference on $x^3$.
- §2.2 The chain rule: the $z = 3x$, $y = z^2$ example (36) and longer pipelines.
- §2.3 Gradients of vector/matrix expressions: the quadratic form and the linear regression gradient $[46, 64]$.
- §2.4 Jacobians: the softmax cross-entropy gradient $p - y$ and log-sum-exp for $z = [1000, 999]$.

numpy only, fixed seeds, runs in a few seconds on a CPU.

## Tools: the numerical gradient and the relative error

`numerical_gradient` raises one coordinate of $x$ by $h$, lowers it by $h$,
evaluates $f$ both times, and divides the difference by $2h$. For $n$
coordinates that costs $2n$ evaluations of $f$, far too slow for training,
but it uses no derivative formula, so it is an independent check on one.
`relative_error` divides the difference by the size of the two values, so a
gradient near 1000 and one near 0.001 are judged on the same scale. A relative
error below $10^{-6}$ counts as a match; with $h = 10^{-5}$ the errors below
land between $10^{-13}$ and $10^{-9}$.

In [1]:
import numpy as np

H = 1e-5    # step size for the centered difference
TOL = 1e-6  # relative error below this counts as a match


def numerical_derivative(f, x, h=H):
    """Centered difference for a scalar function of a scalar."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, x, h=H):
    """Centered difference gradient of f: R^n -> R, one coordinate at a time."""
    x = np.array(x, dtype=float)
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        old = x[idx]
        x[idx] = old + h
        f_plus = f(x)
        x[idx] = old - h
        f_minus = f(x)
        x[idx] = old  # restore
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad


def relative_error(a, b):
    """Largest |a - b| / (|a| + |b|) over all entries; below TOL means a match."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.maximum(np.abs(a) + np.abs(b), 1e-12)
    return np.max(np.abs(a - b) / denom)


def report(name, numerical, analytical, tol=TOL):
    """Print both gradients and their relative error, and assert they match."""
    err = relative_error(numerical, analytical)
    status = "MATCH" if err < tol else "MISMATCH"
    print(name)
    print(f"  numerical : {np.round(np.asarray(numerical), 6)}")
    print(f"  analytical: {np.round(np.asarray(analytical), 6)}")
    print(f"  max relative error = {err:.2e}  [{status}]")
    assert err < tol, f"gradient check failed for {name}"

## 1. Forward vs centered difference (§2.1)

The forward difference $(f(x+h) - f(x))/h$ has an error that shrinks in
proportion to $h$; the centered difference $(f(x+h) - f(x-h))/(2h)$ has an
error that shrinks in proportion to $h^2$. The cell below reproduces the
chapter's table for $f(x) = x^3$ at $x = 1$, where the true derivative is 3.
Expected: forward 3.0301 and 3.003001, centered 3.0001 and 3.000001, so a
tenfold smaller $h$ cuts the forward error 10× and the centered error 100×.

In [2]:
f = lambda x: x ** 3
x0, true = 1.0, 3.0

print(f"{'h':>8s} {'forward':>12s} {'error':>10s} {'centered':>12s} {'error':>10s}")
for h in [1e-2, 1e-3]:
    fwd = (f(x0 + h) - f(x0)) / h
    ctr = (f(x0 + h) - f(x0 - h)) / (2 * h)
    print(f"{h:>8g} {fwd:>12.6f} {abs(fwd - true):>10.6f} "
          f"{ctr:>12.6f} {abs(ctr - true):>10.6f}")

h = 1e-2
assert abs((f(x0 + h) - f(x0)) / h - 3.0301) < 1e-9
assert abs((f(x0 + h) - f(x0 - h)) / (2 * h) - 3.0001) < 1e-9

       h      forward      error     centered      error
    0.01     3.030100   0.030100     3.000100   0.000100
   0.001     3.003001   0.003001     3.000001   0.000001


## 2. Scalar derivatives

Four functions with known derivatives, each checked at one point: $x^2$,
$\sin x$, $e^x$, and the sigmoid $s(x) = 1/(1+e^{-x})$, whose derivative is
$s(x)(1 - s(x))$. The sigmoid formula is the one used in every logistic
regression and in the backward pass of sigmoid layers. All four should print
MATCH.

In [3]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))

cases = [
    # (name, f, f', test point)
    ("f(x) = x^2,  f'(x) = 2x,  at x = 3.0", lambda x: x ** 2, lambda x: 2 * x, 3.0),
    ("f(x) = sin(x),  f'(x) = cos(x),  at x = 1.0", np.sin, np.cos, 1.0),
    ("f(x) = exp(x),  f'(x) = exp(x),  at x = 0.5", np.exp, np.exp, 0.5),
    ("f(x) = sigmoid(x),  f'(x) = s(1 - s),  at x = 2.0",
     sigmoid, lambda x: sigmoid(x) * (1 - sigmoid(x)), 2.0),
]
for name, f, fprime, x0 in cases:
    report(name, numerical_derivative(f, x0), fprime(x0))

f(x) = x^2,  f'(x) = 2x,  at x = 3.0
  numerical : 6.0
  analytical: 6.0
  max relative error = 3.28e-12  [MATCH]
f(x) = sin(x),  f'(x) = cos(x),  at x = 1.0
  numerical : 0.540302
  analytical: 0.540302
  max relative error = 1.03e-11  [MATCH]
f(x) = exp(x),  f'(x) = exp(x),  at x = 0.5
  numerical : 1.648721
  analytical: 1.648721
  max relative error = 5.86e-12  [MATCH]
f(x) = sigmoid(x),  f'(x) = s(1 - s),  at x = 2.0
  numerical : 0.104994
  analytical: 0.104994
  max relative error = 5.02e-13  [MATCH]


## 3. The chain rule: rates multiply along a pipeline (§2.2)

For $z = g(x)$ followed by $y = f(z)$, the chain rule says
$dy/dx = (dy/dz)(dz/dx)$. The first check is the chapter's example
$z = 3x$, $y = z^2$ at $x = 2$: the local rates are 3 and 12, their product is
36, and the numerical derivative of $y = 9x^2$ at $x = 2$ is also 36. The
second check uses $z = x^2 + 1$, $y = \sin z$ at $x = 0.7$ and compares the
product of the two measured local rates against the measured rate of the
whole composition.

In [4]:
# Chapter example: z = 3x, y = z^2, at x = 2.
x0 = 2.0
z0 = 3 * x0
dz_dx, dy_dz = 3.0, 2 * z0
print(f"z = 3x, y = z^2 at x = {x0}: dz/dx = {dz_dx}, dy/dz = 2z = {dy_dz}, "
      f"product = {dy_dz * dz_dx}")
report("dy/dx for y = (3x)^2 at x = 2", numerical_derivative(lambda x: (3 * x) ** 2, x0),
       dy_dz * dz_dx)
assert dy_dz * dz_dx == 36.0
print()

# z = x^2 + 1, y = sin(z), at x = 0.7: every rate measured numerically.
g = lambda x: x ** 2 + 1
f = lambda z: np.sin(z)
x0 = 0.7
z0 = g(x0)
dz_dx = numerical_derivative(g, x0)                    # inner rate
dy_dz = numerical_derivative(f, z0)                    # outer rate
dy_dx = numerical_derivative(lambda x: f(g(x)), x0)    # rate of the composition
print(f"z = x^2 + 1, y = sin(z) at x = {x0} (z = {z0:.6f})")
print(f"  dz/dx, measured           = {dz_dx:.8f}")
print(f"  dy/dz, measured           = {dy_dz:.8f}")
print(f"  product dy/dz * dz/dx     = {dy_dz * dz_dx:.8f}")
print(f"  dy/dx of the composition  = {dy_dx:.8f}")
print(f"  formula cos(x^2 + 1) * 2x = {np.cos(x0 ** 2 + 1) * 2 * x0:.8f}")
err = relative_error(dy_dz * dz_dx, dy_dx)
print(f"  product vs composition: relative error = {err:.2e}  "
      f"[{'MATCH' if err < 1e-5 else 'MISMATCH'}]")
assert err < 1e-5

z = 3x, y = z^2 at x = 2.0: dz/dx = 3.0, dy/dz = 2z = 12.0, product = 36.0
dy/dx for y = (3x)^2 at x = 2
  numerical : 36.0
  analytical: 36.0
  max relative error = 1.07e-11  [MATCH]

z = x^2 + 1, y = sin(z) at x = 0.7 (z = 1.490000)
  dz/dx, measured           = 1.40000000
  dy/dz, measured           = 0.08070845
  product dy/dz * dz/dx     = 0.11299183
  dy/dx of the composition  = 0.11299183
  formula cos(x^2 + 1) * 2x = 0.11299183
  product vs composition: relative error = 6.72e-10  [MATCH]


## 4. A three-step pipeline, the shape of a tiny network

The pipeline is $x \to a = 3x \to b = \text{sigmoid}(a) \to y = b^2$. Each
step contributes one local derivative computed from its own input and output
only: 3, $b(1-b)$, and $2b$. Backpropagation multiplies these local
derivatives along the path. The product should equal the numerical derivative
of the whole pipeline at $x = 0.2$.

In [5]:
x0 = 0.2
a0 = 3 * x0
b0 = sigmoid(a0)
local = [
    ("da/dx = 3", 3.0),
    ("db/da = b(1 - b)", b0 * (1 - b0)),
    ("dy/db = 2b", 2 * b0),
]
prod = 1.0
for name, val in local:
    prod *= val
    print(f"local {name:18s} = {val:.8f}")
direct = numerical_derivative(lambda x: sigmoid(3 * x) ** 2, x0)
print(f"product of the three        = {prod:.8f}")
print(f"numerical dy/dx of pipeline = {direct:.8f}")
err = relative_error(prod, direct)
print(f"relative error = {err:.2e}  [{'MATCH' if err < 1e-5 else 'MISMATCH'}]")
assert err < 1e-5

local da/dx = 3          = 3.00000000
local db/da = b(1 - b)   = 0.22878424
local dy/db = 2b         = 1.29131261
product of the three        = 0.88629593
numerical dy/dx of pipeline = 0.88629593
relative error = 4.57e-11  [MATCH]


## 5. Gradient of the quadratic form $x^\top A x$ (§2.3)

The identity is $\nabla_x\, x^\top A x = (A + A^\top)x$, the vector version of
$ax^2 \to 2ax$. The matrix below is deliberately not symmetric, so the common
guess $2Ax$ gives the wrong answer. These are the numbers of exercise 5 in
[exercises.md](exercises.md): $f(x) = 16$ and the gradient is $[6, 13]$.

In [6]:
A = np.array([[2.0, 1.0],
              [0.0, 3.0]])  # not symmetric
x0 = np.array([1.0, 2.0])
f = lambda x: x @ A @ x
print(f"A =\n{A}\nx = {x0},  f(x) = x^T A x = {f(x0)}\n")

report("grad of x^T A x, non-symmetric A", numerical_gradient(f, x0), (A + A.T) @ x0)
print(f"\nThe guess 2Ax gives {2 * A @ x0}, which is wrong unless A is symmetric.")
assert np.allclose((A + A.T) @ x0, [6, 13])

A =
[[2. 1.]
 [0. 3.]]
x = [1. 2.],  f(x) = x^T A x = 16.0

grad of x^T A x, non-symmetric A
  numerical : [ 6. 13.]
  analytical: [ 6. 13.]
  max relative error = 1.18e-11  [MATCH]

The guess 2Ax gives [ 8. 12.], which is wrong unless A is symmetric.


## 6. The linear regression gradient (§2.3 worked example)

For $L(w) = \|Xw - y\|^2$ the gradient is $2X^\top(Xw - y)$: each weight's
entry is twice the sum over examples of (error × that example's feature
value). The first check reproduces the chapter's numbers,
$X = [[1, 2], [3, 4]]$, $y = [1, 0]$, $w = [1, 1]$, where the errors are
$[2, 7]$ and the gradient is $[46, 64]$. The second check uses random data
with 6 examples and 3 features, then solves the normal equations
$X^\top X w = X^\top y$ and confirms the gradient there is 0.

In [7]:
# Chapter example.
X = np.array([[1.0, 2.0],
              [3.0, 4.0]])
y = np.array([1.0, 0.0])
w = np.array([1.0, 1.0])
loss = lambda w: np.sum((X @ w - y) ** 2)
r = X @ w - y
print(f"predictions Xw = {X @ w}, errors r = {r}, L(w) = {loss(w)}")
analytical = 2 * X.T @ r
report("grad_w ||Xw - y||^2, chapter example", numerical_gradient(loss, w), analytical)
assert np.allclose(analytical, [46, 64])
print()

# Random data: 6 examples, 3 features.
rng = np.random.default_rng(42)
n, d = 6, 3
X = rng.standard_normal((n, d))
true_w = np.array([1.5, -2.0, 0.5])
y = X @ true_w + 0.1 * rng.standard_normal(n)  # targets with a little noise
w0 = rng.standard_normal(d)                   # an arbitrary weight vector
loss = lambda w: np.sum((X @ w - y) ** 2)
print(f"X shape {X.shape}, w = {np.round(w0, 4)}, L(w) = {loss(w0):.6f}")
report("grad_w ||Xw - y||^2, random data", numerical_gradient(loss, w0),
       2 * X.T @ (X @ w0 - y))

# At the least-squares solution the gradient is zero.
w_star = np.linalg.solve(X.T @ X, X.T @ y)
grad_norm = np.linalg.norm(2 * X.T @ (X @ w_star - y))
print(f"\nnormal-equations solution w* = {np.round(w_star, 4)} (true w = {true_w})")
print(f"||gradient at w*|| = {grad_norm:.2e}")
assert grad_norm < 1e-10

predictions Xw = [3. 7.], errors r = [2. 7.], L(w) = 53.0
grad_w ||Xw - y||^2, chapter example
  numerical : [46. 64.]
  analytical: [46. 64.]
  max relative error = 4.24e-12  [MATCH]

X shape (6, 3), w = [-0.4283 -0.3521  0.5323], L(w) = 49.669567
grad_w ||Xw - y||^2, random data
  numerical : [-20.251248  36.084179  11.831331]
  analytical: [-20.251248  36.084179  11.831331]
  max relative error = 3.38e-11  [MATCH]

normal-equations solution w* = [ 1.586  -1.9698  0.5204] (true w = [ 1.5 -2.   0.5])
||gradient at w*|| = 3.65e-15


## 7. Softmax followed by cross-entropy: the gradient is $p - y$ (§2.4)

With logits $z$, probabilities $p = \text{softmax}(z)$, and one-hot target
$y$, the loss is $L = -\log p_c = -z_c + \log \sum_j e^{z_j}$, and its
gradient with respect to $z$ is $p - y$. The cell uses the chapter's example,
$z = [2, 0, 1]$ with true class 1: expected $p \approx [0.67, 0.09, 0.24]$,
$L \approx 0.41$, and gradient $\approx [-0.33, 0.09, 0.24]$. It also computes
the same gradient step by step, through $\partial L/\partial p = -y/p$ and the
dense softmax Jacobian $J_{ij} = p_i(\delta_{ij} - p_j)$, and then repeats the
check on random logits with 5 classes.

In [8]:
def softmax(z):
    e = np.exp(z - np.max(z))  # subtracting the max avoids overflow (next section)
    return e / e.sum()


def cross_entropy(z, c):
    return -np.log(softmax(z)[c])


z = np.array([2.0, 0.0, 1.0])
c = 0                      # true class 1 is index 0
y = np.eye(3)[c]
p = softmax(z)
print(f"z = {z}, y = {y}")
print(f"p = {np.round(p, 2)}, L = -log p_1 = {cross_entropy(z, c):.2f}")
report("dL/dz for softmax + cross-entropy, chapter example",
       numerical_gradient(lambda z: cross_entropy(z, c), z), p - y)

# Step by step: dL/dp = -y/p, then multiply by the dense softmax Jacobian.
J = np.diag(p) - np.outer(p, p)
step_by_step = J.T @ (-y / p)
print(f"step by step through J: {np.round(step_by_step, 6)}")
assert np.allclose(step_by_step, p - y)
print()

# Random logits, 5 classes.
rng = np.random.default_rng(0)
z = rng.standard_normal(5) * 3
c = 3
report("dL/dz for softmax + cross-entropy, random 5-class logits",
       numerical_gradient(lambda z: cross_entropy(z, c), z), softmax(z) - np.eye(5)[c])

z = [2. 0. 1.], y = [1. 0. 0.]
p = [0.67 0.09 0.24], L = -log p_1 = 0.41
dL/dz for softmax + cross-entropy, chapter example
  numerical : [-0.334759  0.090031  0.244728]
  analytical: [-0.334759  0.090031  0.244728]
  max relative error = 6.69e-12  [MATCH]
step by step through J: [-0.334759  0.090031  0.244728]

dL/dz for softmax + cross-entropy, random 5-class logits
  numerical : [ 0.138467  0.063888  0.648529 -0.869921  0.019038]
  analytical: [ 0.138467  0.063888  0.648529 -0.869921  0.019038]
  max relative error = 5.73e-10  [MATCH]


## 8. Log-sum-exp: softmax without overflow (§2.4)

For logits $z = [1000, 999]$, computing $e^{1000}$ overflows a 64-bit float
(the largest is about $e^{709}$), so the direct formulas return `inf` and
`nan`. Subtracting the largest logit $m$ first gives the same softmax and
keeps every exponent at most $e^0 = 1$:
$\log \sum_j e^{z_j} = m + \log \sum_j e^{z_j - m}$. Expected:
$p \approx [0.731, 0.269]$ and $\log \sum_j e^{z_j} \approx 1000.313$. The
last check confirms that the gradient of log-sum-exp is the softmax, which is
where the $p$ in $p - y$ comes from.

In [9]:
z = np.array([1000.0, 999.0])

with np.errstate(over="ignore", invalid="ignore"):
    naive_softmax = np.exp(z) / np.exp(z).sum()
    naive_lse = np.log(np.exp(z).sum())
print(f"direct formulas:     softmax = {naive_softmax},  log sum exp = {naive_lse}")


def logsumexp(z):
    m = np.max(z)
    return m + np.log(np.sum(np.exp(z - m)))


m = z.max()
p_stable = np.exp(z - m) / np.exp(z - m).sum()
lse = logsumexp(z)
log_p = z - lse
print(f"after subtracting m: softmax = {np.round(p_stable, 3)},  log sum exp = {lse:.3f}")
print(f"log p = z - logsumexp(z) = {np.round(log_p, 4)}")
assert np.allclose(p_stable, [0.731, 0.269], atol=5e-4)
assert abs(lse - (1000 + np.log1p(np.exp(-1)))) < 1e-9
assert abs(lse - 1000.313) < 5e-4
print()

report("gradient of logsumexp at z = [1000, 999] equals softmax(z)",
       numerical_gradient(logsumexp, z), p_stable)

direct formulas:     softmax = [nan nan],  log sum exp = inf
after subtracting m: softmax = [0.731 0.269],  log sum exp = 1000.313
log p = z - logsumexp(z) = [-0.3133 -1.3133]

gradient of logsumexp at z = [1000, 999] equals softmax(z)
  numerical : [0.731059 0.268941]
  analytical: [0.731059 0.268941]
  max relative error = 2.09e-09  [MATCH]


Every cell above ends in an `assert`, so reaching this point means all
gradient checks passed.